# 01 — Exploratory Data Analysis

**Belgium Campus · Student Academic Risk Prediction**

Each section answers **one question**. Strong predictors for this model are attendance, assignment completion, test/midyear averages, BC Connect activity, missed assessments, and failed modules — not programme alone.

| # | Question |
|---|----------|
| 1 | What does the dataset even look like? |
| 2 | How big is our dataset? |
| 3 | What type of information is inside? |
| 4 | Are there missing values? |
| 5 | Are there duplicate students? |
| 6 | Is our target balanced? |
| 7 | Does attendance look realistic? |
| 8 | Does overall average look believable? |
| 9 | Does attendance affect risk? |
| 10 | Does overall average affect risk? |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "data").exists() else NOTEBOOK_DIR.parent
DATA_PATH = PROJECT_ROOT / "data" / "synthetic" / "students.csv"

df = pd.read_csv(DATA_PATH)

RISK_ORDER = ["Low", "Medium", "High"]
RISK_COLORS = {"Low": "#2a9d8f", "Medium": "#e9c46a", "High": "#e76f51"}

print(f"Loaded: {DATA_PATH}")
print(f"Ready to investigate {len(df):,} student records.")

---
## Q1 — What does the dataset even look like?

In [ ]:
display(df.head())
display(df.iloc[0].to_frame(name="value"))
print("Programme × specialisation sample:")
display(df.groupby(["programme", "specialisation"]).size().rename("n").reset_index().head(12))

**Finding:** Rows are anonymised BC students with `programme` + `specialisation`, year labels like `First Year`, engagement predictors, and % academic averages (no GPA).

---
## Q2 — How big is our dataset?

In [ ]:
n_rows, n_cols = df.shape
display(
    pd.DataFrame(
        {
            "metric": ["rows (students)", "columns", "memory (KB)"],
            "value": [n_rows, n_cols, round(df.memory_usage(deep=True).sum() / 1024, 1)],
        }
    )
)
display(df["programme"].value_counts().rename("count").to_frame())
display(df["year_of_study"].value_counts().rename("count").to_frame())

**Finding:** 2,000 students across Belgium Campus programmes and year bands.

---
## Q3 — What type of information is inside?

In [ ]:
display(
    pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "non_null": df.notna().sum(),
            "n_unique": df.nunique(),
            "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
        }
    )
)
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(exclude="number").columns.tolist()
print(f"Numeric: {numeric_cols}")
print(f"Categorical / ID: {categorical_cols}")
display(df[numeric_cols].describe().round(2))

**Finding:** Key model inputs are mostly numeric predictors (`attendance`, `assignment_completion`, `test_average`, `bc_connect_activity`, `missed_assessments`, `failed_modules`). `programme` / `specialisation` / `year_of_study` are categorical context.

---
## Q4 — Are there missing values?

In [ ]:
missing = pd.DataFrame(
    {"missing_count": df.isna().sum(), "missing_pct": (df.isna().mean() * 100).round(2)}
).sort_values("missing_count", ascending=False)
display(missing)
print(f"Total missing cells: {int(df.isna().sum().sum())}")

fig, ax = plt.subplots(figsize=(10, 4))
missing["missing_pct"].plot(kind="bar", ax=ax, color="#457b9d", edgecolor="white")
ax.set_title("Missing values by column (%)")
ax.set_ylabel("% missing")
ax.set_ylim(0, max(5, missing["missing_pct"].max() * 1.2))
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

**Finding:** No missing values.

---
## Q5 — Are there duplicate students?

In [ ]:
display(
    pd.DataFrame(
        {
            "check": ["unique student_id", "duplicate IDs", "identical rows", "n rows"],
            "value": [
                df["student_id"].nunique(),
                int(df["student_id"].duplicated().sum()),
                int(df.duplicated().sum()),
                len(df),
            ],
        }
    )
)
assert df["student_id"].nunique() == len(df)

**Finding:** One row per `student_id`.

---
## Q6 — Is our target balanced?

In [ ]:
target_counts = df["risk_label"].value_counts().reindex(RISK_ORDER)
target_pct = (df["risk_label"].value_counts(normalize=True).reindex(RISK_ORDER) * 100).round(1)
display(pd.DataFrame({"count": target_counts, "percent": target_pct}))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = [RISK_COLORS[l] for l in RISK_ORDER]
axes[0].bar(RISK_ORDER, target_counts, color=colors, edgecolor="white")
axes[0].set_title("Target counts")
axes[1].pie(target_counts, labels=RISK_ORDER, autopct="%1.1f%%", colors=colors, startangle=90,
            wedgeprops={"edgecolor": "white"})
axes[1].set_title("Target share")
plt.tight_layout()
plt.show()

**Finding:** Classes are roughly balanced.

---
## Q7 — Does attendance look realistic?

In [ ]:
att = df["attendance"]
display(att.describe().round(1).to_frame("attendance"))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(att, bins=20, color="#457b9d", edgecolor="white")
axes[0].set_title("Attendance (%)")
axes[0].set_xlabel("Attendance %")
df.boxplot(column="attendance", by="profile", ax=axes[1])
axes[1].set_title("Attendance by profile")
plt.suptitle("")
plt.tight_layout()
plt.show()

**Finding:** Attendance is an integer % in a believable campus range.

---
## Q8 — Does overall average look believable?

In [ ]:
avg = df["overall_average"]
display(avg.describe().round(1).to_frame("overall_average"))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(avg, bins=20, color="#2a9d8f", edgecolor="white")
axes[0].set_title("Overall average (%)")
df.boxplot(column="overall_average", by="profile", ax=axes[1])
axes[1].set_title("Overall average by profile")
plt.suptitle("")
plt.tight_layout()
plt.show()

print("Related mark columns:")
display(df[["midyear_average", "test_average", "assignment_average", "practical_average"]].describe().round(1))

**Finding:** Marks use a 0–100% scale (`midyear_average`, `test_average`, etc.) — SA / Belgium Campus style, not GPA.

---
## Q9 — Does attendance affect risk?

In [ ]:
att_by_risk = (
    df.groupby("risk_label")["attendance"].agg(count="size", mean="mean", median="median")
    .reindex(RISK_ORDER).round(1)
)
display(att_by_risk)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
data = [df.loc[df["risk_label"] == l, "attendance"] for l in RISK_ORDER]
bp = axes[0].boxplot(data, tick_labels=RISK_ORDER, patch_artist=True)
for patch, label in zip(bp["boxes"], RISK_ORDER):
    patch.set_facecolor(RISK_COLORS[label])
axes[0].set_title("Attendance by risk")
axes[1].bar(RISK_ORDER, att_by_risk["mean"], color=[RISK_COLORS[l] for l in RISK_ORDER], edgecolor="white")
axes[1].set_title("Mean attendance by risk")
plt.tight_layout()
plt.show()

risk_ord = df["risk_label"].map({"Low": 0, "Medium": 1, "High": 2})
print(f"corr(attendance, risk): {df['attendance'].corr(risk_ord):.3f}")

**Finding:** Lower attendance ↔ higher risk.

---
## Q10 — Does overall average affect risk?

In [ ]:
avg_by_risk = (
    df.groupby("risk_label")["overall_average"].agg(count="size", mean="mean", median="median")
    .reindex(RISK_ORDER).round(1)
)
display(avg_by_risk)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
data = [df.loc[df["risk_label"] == l, "overall_average"] for l in RISK_ORDER]
bp = axes[0].boxplot(data, tick_labels=RISK_ORDER, patch_artist=True)
for patch, label in zip(bp["boxes"], RISK_ORDER):
    patch.set_facecolor(RISK_COLORS[label])
axes[0].set_title("Overall average by risk")
axes[1].bar(RISK_ORDER, avg_by_risk["mean"], color=[RISK_COLORS[l] for l in RISK_ORDER], edgecolor="white")
axes[1].set_title("Mean overall average by risk")
plt.tight_layout()
plt.show()

risk_ord = df["risk_label"].map({"Low": 0, "Medium": 1, "High": 2})
print(f"corr(overall_average, risk): {df['overall_average'].corr(risk_ord):.3f}")
print("Other strong predictors vs risk:")
for col in ["assignment_completion", "test_average", "bc_connect_activity", "missed_assessments", "failed_modules"]:
    print(f"  corr({col}, risk): {df[col].corr(risk_ord):.3f}")

**Finding:** Lower overall average (and weaker engagement/assessment signals) track with higher risk.

---
## EDA takeaways

| Question | Verdict |
|----------|---------|
| Shape | BC programmes + specialisations; % marks; engagement predictors |
| Size | 2,000 students |
| Missing / duplicates | None |
| Target | Roughly balanced |
| Main risk drivers | Attendance, completion, tests/midyear, BC Connect, missed assessments, fails |